# 05 — Big Data Techniques

Demonstrates the four non-Spark big-data techniques from `docs/big_data.md`: chunked batch processing, a streaming simulation over the monthly rates time series, and CPU-parallelism benchmarking. The PySpark job itself (`src/spark/spark_train.py`) needs a Spark runtime and is run separately via `spark-submit` — see the last cell for the command.

In [ ]:
import sys
sys.path.insert(0, "..")

import matplotlib.pyplot as plt
import pandas as pd

from src.spark.batch_processing import run_batch_processing
from src.spark.stream_processing import run_streaming_simulation
from src.utils.config import resolve_path
from src.visualization.plots import save_figure, set_publication_style

set_publication_style()

## 1. Batch processing — chunked, out-of-core statistics

In [ ]:
from src.utils.io import read_table, write_table
df = read_table(resolve_path("processed_dataset"))
batch_df = run_batch_processing(df, chunk_size=10_000)
print(f"Processed {len(batch_df)} batches in {batch_df.attrs['elapsed_seconds']:.2f}s")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(batch_df["batch"], batch_df["mean_revenue"], marker="o", color="steelblue")
ax.axhline(batch_df["mean_revenue"].mean(), color="red", linestyle="--",
           label=f"Overall mean: ${batch_df['mean_revenue'].mean():,.0f}")
ax.set_title("Batch Processing: Revenue Stability Across Batches")
ax.set_xlabel("Batch number"); ax.set_ylabel("Mean revenue (USD)"); ax.legend()
save_figure(fig, resolve_path("figure_dir") / "figure12_batch_processing.png")

## 2. Streaming simulation — month-by-month over `past_rates.parquet`

In [ ]:
from src.utils.io import read_table, write_table
rates = read_table(resolve_path("raw_past_rates"))
stream_df = run_streaming_simulation(rates)
print(f"Streamed {len(stream_df)} monthly periods in {stream_df.attrs['elapsed_seconds']:.2f}s")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(stream_df["period"].astype(str), stream_df["total_revenue"] / 1e6,
        marker="o", color="seagreen")
ax.set_title("Streaming Pipeline: Monthly Revenue Trend")
ax.set_ylabel("Total revenue (M USD)")
plt.xticks(rotation=45, ha="right", fontsize=7)
save_figure(fig, resolve_path("figure_dir") / "figure13_streaming_trend.png")

## 3. CPU multi-core parallelism

In [ ]:
import time
import multiprocessing
from sklearn.ensemble import RandomForestRegressor

from src.features.feature_engineer import FeatureEngineer
from src.utils.config import load_training_config

cfg = load_training_config()
engineered = FeatureEngineer(top_n_countries=cfg["top_n_countries"]).fit_transform(df)
X = engineered[cfg["numeric_features"]].fillna(engineered[cfg["numeric_features"]].median())
y = engineered[cfg["target"]]

n_cores = multiprocessing.cpu_count()
for n_jobs in [1, max(2, n_cores // 2), -1]:
    t0 = time.time()
    RandomForestRegressor(n_estimators=100, max_depth=10, n_jobs=n_jobs, random_state=42).fit(X, y)
    label = f"n_jobs={n_jobs}" if n_jobs != -1 else f"n_jobs=-1 (all {n_cores} cores)"
    print(f"{label:30s} -> {time.time() - t0:.2f}s")

## 4. PySpark distributed pipeline (run separately)

```bash
spark-submit src/spark/spark_train.py --input data/processed/data_clean.parquet
```

See `docs/big_data.md` for what this job does and the GPU acceleration strategy for Oracle VM deployment.